# 07 — Golden Files and C++ Agreement Checks

Notebooks are only useful if their expected answers match the real compiled library. Crankl keeps fixed reference files under `tests/golden/` and checks them from C++ tests.

This notebook:

1. Runs `scripts/export_golden.py` to refresh those reference files (same as CI)
2. Shows the generated manifest and algebra references
3. Recomputes the same Clifford checks through the public C API
4. Confirms expected vs actual match (tiny floating-point tolerance)

If something fails here, the reference data and the production kernels have drifted apart.

In [ ]:
from pathlib import Path
import sys
import json
import subprocess

root = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
notebook_dir = root / "notebooks"
sys.path.insert(0, str(notebook_dir)) if str(notebook_dir) not in sys.path else None

from crankl_demo import CranklAPI, Multivector

export = subprocess.run(
    [sys.executable, str(root / "scripts" / "export_golden.py")],
    cwd=root,
    check=True,
    text=True,
    capture_output=True,
)
print(export.stdout.strip())

manifest_path = root / "tests" / "golden" / "manifest.json"
manifest = json.loads(manifest_path.read_text())
print("\nGolden manifest:")
print(json.dumps(manifest, indent=2))

## Recheck the algebra identities through the shared library

`algebra_ref.json` stores the expected answers for a few basic products (like `e1 * e1` and `e1 * e2`).

The next cell computes the same values with the C API and prints **expected vs actual**. They should match within `1e-9`.

In [ ]:
api = CranklAPI()
e1 = Multivector.create(vector=(1.0, 0.0, 0.0))
e2 = Multivector.create(vector=(0.0, 1.0, 0.0))
e12 = Multivector.create(bivector=(1.0, 0.0, 0.0))

actual = {
    "e1_e1_scalar": api.clifford_product(e1, e1).s,
    "e1_e2_b12": api.clifford_product(e1, e2).b[0],
    "e12_e12_scalar": api.clifford_product(e12, e12).s,
    "e1_e2_e12_scalar": api.clifford_product(api.clifford_product(e1, e2), e12).s,
}
expected = json.loads((root / "tests" / "golden" / "algebra_ref.json").read_text())

for name in expected:
    print(f"{name:24s} expected={expected[name]: .6f} actual={actual[name]: .6f}")
    assert abs(expected[name] - actual[name]) < 1e-9